# ShopFlow \u2014 Olist Data Exploration

Exploratory data analysis on the Brazilian Olist e-commerce dataset.

Per VISION.md §5, historical Olist data is used **only for ML training**, kept separate from live PostgreSQL marketplace data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")

DATA_DIR = Path("../../dataset")
print("Dataset files:", [p.name for p in DATA_DIR.glob("*.csv")])

In [ ]:
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "translation": translation,
}

for name, df in tables.items():
    print(f"{name:18s}: {df.shape[0]:>6,} rows x {df.shape[1]:>2} cols")

In [ ]:
orders["order_status"].value_counts().plot(kind="bar", figsize=(8,4), title="Order Status Distribution")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400
orders["is_late"] = (orders["delivery_delay_days"] > 0).astype(int)

print(f"Late delivery rate: {orders['is_late'].mean():.2%}")
print(f"Avg delay (days): {orders['delivery_delay_days'].mean():.2f}")

orders["delivery_delay_days"].plot(kind="hist", bins=50, figsize=(8,4), title="Delivery Delay Distribution (days)")
plt.xlabel("Delay days")
plt.tight_layout()
plt.show()

In [ ]:
reviews["review_score"].value_counts().sort_index().plot(kind="bar", figsize=(8,4), title="Review Score Distribution")
plt.xlabel("Review Score")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
items_pay = order_items.merge(payments[['order_id', 'payment_value']], on="order_id", how="left")
cat_rev = items_pay.groupby("product_id")["price"].sum().reset_index()
cat_rev = cat_rev.merge(products[['product_id', 'product_category_name']], on="product_id")
cat_rev = cat_rev.merge(translation, on="product_category_name", how="left")

top_cats = cat_rev.groupby("product_category_name_english")["price"].sum().sort_values(ascending=False).head(10)
top_cats.plot(kind="barh", figsize=(10,6), title="Top 10 Product Categories by Revenue")
plt.xlabel("Revenue (BRL)")
plt.tight_layout()
plt.show()